# Transformacion  de Datos: Silver a Gold
Encargado: Charloth Salgado
Se tomaran los datos limpios del archivo students_limpios.csv y aplicaremos reglas utilizando **PySpark**. El objetivo es generar tablas optimizadas que seran guardadas en formato Parquet, para ser consultados por parte del equipo de analisis.

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum, max, when

spark =SparkSession.builder.appName("Transformacion_zona_gold").getOrCreate() 
print("sesion iniciada - Spark")

print("Leyendo datos...")

entrada= "../data/processed/students_limpios.csv"
limpio= spark.read.csv(entrada, header=True, inferSchema=True)
print("Datos leidos correctamente")

sesion iniciada - Spark
Leyendo datos...


Datos leidos correctamente


# Vista del esquema
Antes de la transformacion se verifica que PySpark infiriera correctamente los tipos de datos a partir del CSV limpio.

In [10]:
limpio.printSchema()
limpio.show(5)

root
 |-- Student_ID: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- First_Generation: integer (nullable = true)
 |-- Family_Income: double (nullable = true)
 |-- Household_Size: integer (nullable = true)
 |-- Housing_Status: string (nullable = true)
 |-- Scholarship: integer (nullable = true)
 |-- Tuition_Base: integer (nullable = true)
 |-- Semester: integer (nullable = true)
 |-- Course_Load: integer (nullable = true)
 |-- Work_Hours: double (nullable = true)
 |-- Emergency_Expense: double (nullable = true)
 |-- Sem_GPA: double (nullable = true)
 |-- Attendance: double (nullable = true)
 |-- LMS_Logins: double (nullable = true)
 |-- Advising_Visits: integer (nullable = true)
 |-- Failed_Courses: integer (nullable = true)
 |-- Financial_Stress: double (nullable = true)
 |-- Target_Dropout_Next_Sem: integer (nullable = true)
 |-- End_of_Semester_Status: string (nullable = true)
 |-- Censored: integer (nullable = true)
 |-- inges

# Aplicacion y vista de Transformaciones 
Se aplicaran dos trasformaciones.
1- Clasificacion de Riesgo - columna calculada: 
se crea un columna llamada Estado_Riesgo, donde se verifica si el promedio de notas del semestre **Sem_GPA** es menor a 2.0, el estudiante se marcara como "En Riesgo" si no "Estable"
2- Resumen historico - agrupacion general:
Se agruparan los datos por estudiente, para obtener un resumen general de su vida academica (promedio de notas, asistencia, total de cursos y ultimo semestre cursado)

In [4]:
print("Aplicando Transformaciones...")
df_trans = limpio.withColumn(
    "Estado_Riesgo",
    when(col("Sem_GPA") < 2.0, "En Riesgo").otherwise("Estable")
)

df_gold = df_trans.groupBy("Student_ID").agg(
    avg("Sem_GPA").alias("Promedio_GPA_Historico"),
    avg("Attendance").alias("Promedio_Asistencia"),
    sum("Course_Load").alias("Total_Cursos_Tomados"),
    max("Semester").alias("Ultimo_Semestre_Cursado")
)
print("-----Transformaciones Exitosas-----")

df_gold.show(10)

Aplicando Transformaciones...
-----Transformaciones Exitosas-----


+----------+----------------------+-------------------+--------------------+-----------------------+
|Student_ID|Promedio_GPA_Historico|Promedio_Asistencia|Total_Cursos_Tomados|Ultimo_Semestre_Cursado|
+----------+----------------------+-------------------+--------------------+-----------------------+
| STU_00160|     3.016666666666666|  81.18333333333334|                  90|                      6|
| STU_00323|    2.9400000000000004|  89.64999999999999|                  66|                      4|
| STU_00361|                  3.03|               92.9|                  18|                      1|
| STU_00365|                  2.51|               91.2|                  15|                      1|
| STU_00486|                  3.03|               80.7|                  18|                      1|
| STU_00503|    3.0716666666666668|  83.83333333333333|                  90|                      6|
| STU_01108|                  2.79|               78.1|                  27|               

# Almacenamiento en Parquet
se guarda el DataFrame en el archivo resumen_estudiastes en la carpeta data/gold/, se ultiza el formato **Parquet** que permite la compresion y consultas mas rapidas que en un archivo CSV. 

In [8]:
salida = "../data/gold/resumen_estudiantes"
df_gold.write.mode("overwrite").parquet(salida)
print(" Datos guardados en archivo Parquet resumen_estudiantes")

 Datos guardados en archivo Parquet resumen_estudiantes


# Comprobacion archivo Parquet
Comprobacion de que los datos se guardaran correctamente y la lectura del archivo

In [9]:
df_prueba = spark.read.parquet(salida)
print("Lectura de Parquet exitosa. Mostrando datos:")
df_prueba.show(5)

Lectura de Parquet exitosa. Mostrando datos:
+----------+----------------------+-------------------+--------------------+-----------------------+
|Student_ID|Promedio_GPA_Historico|Promedio_Asistencia|Total_Cursos_Tomados|Ultimo_Semestre_Cursado|
+----------+----------------------+-------------------+--------------------+-----------------------+
| STU_00160|     3.016666666666666|  81.18333333333334|                  90|                      6|
| STU_00323|    2.9400000000000004|  89.64999999999999|                  66|                      4|
| STU_00361|                  3.03|               92.9|                  18|                      1|
| STU_00365|                  2.51|               91.2|                  15|                      1|
| STU_00486|                  3.03|               80.7|                  18|                      1|
+----------+----------------------+-------------------+--------------------+-----------------------+
only showing top 5 rows
